# Repeatable RLSE CUDA validation

This notebook calls `scripts/gpu_validation.py`; model execution, custom CUDA kernel validation (`scripts/validate_cuda_kernels.py`), and live HTTP server validation (`scripts/validate_gpu_server.py`) live in the repository.

### Option A: VS Code Local → Colab Runtime (Zero-Popup Fast Sync)
On your local checkout, run:
```bash
make colab-sync
# or: python3 scripts/gpu_validation.py pack --output /tmp/rlse-source.tar.gz --embed-notebook
```
Then run all cells below directly in VS Code connected to your Colab GPU runtime. Before committing, run `python3 scripts/gpu_validation.py clear-notebook` to strip the embedded archive.

### Option B: Browser Colab Upload
On your local checkout, run:
```bash
python3 scripts/gpu_validation.py pack --output /tmp/rlse-source.tar.gz
```
When `EMBEDDED_ARCHIVE_B64` is empty, Cell 2 prompts you to upload `/tmp/rlse-source.tar.gz`, `/tmp/rlse-source.tar.gz.json`, and `scripts/gpu_validation.py`.


In [ ]:
import os
from pathlib import Path
import shutil
import subprocess

for bin_dir in (Path.home() / ".cargo/bin", Path("/usr/local/cuda/bin")):
    if bin_dir.is_dir() and str(bin_dir) not in os.environ.get("PATH", "").split(os.pathsep):
        os.environ["PATH"] = f"{bin_dir}{os.pathsep}{os.environ.get('PATH', '')}"

subprocess.run(["nvidia-smi"], check=False)
subprocess.run(["nvcc", "--version"], check=False)

if not all(shutil.which(tool) for tool in ("cargo", "rustc", "rustfmt", "cargo-clippy")):
    print("Bootstrapping Rust toolchain (minimal + rustfmt + clippy)...", flush=True)
    subprocess.run(
        "curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal -c rustfmt -c clippy",
        shell=True,
        check=True,
    )
    cargo_bin = str(Path.home() / ".cargo/bin")
    if cargo_bin not in os.environ.get("PATH", "").split(os.pathsep):
        os.environ["PATH"] = f"{cargo_bin}{os.pathsep}{os.environ.get('PATH', '')}"

subprocess.run(["cargo", "--version"], check=True)
subprocess.run(["rustc", "--version"], check=True)


In [ ]:
import base64
import hashlib
import json
from pathlib import Path
import subprocess
import sys
import uuid

EMBEDDED_ARCHIVE_B64 = "..."
EMBEDDED_METADATA_B64 = "..."
EMBEDDED_SCRIPT_B64 = "..."

required = {"rlse-source.tar.gz", "rlse-source.tar.gz.json", "gpu_validation.py"}
if EMBEDDED_ARCHIVE_B64 and EMBEDDED_METADATA_B64 and EMBEDDED_SCRIPT_B64:
    uploaded = {
        "rlse-source.tar.gz": base64.b64decode(EMBEDDED_ARCHIVE_B64),
        "rlse-source.tar.gz.json": base64.b64decode(EMBEDDED_METADATA_B64),
        "gpu_validation.py": base64.b64decode(EMBEDDED_SCRIPT_B64),
    }
    print("Using embedded local-mode archive payload.")
else:
    from google.colab import files
    uploaded = files.upload()
    if not required.issubset(uploaded):
        raise ValueError(f"Upload exactly named files: {sorted(required)}")

metadata = json.loads(uploaded["rlse-source.tar.gz.json"])
bootstrap = uploaded["gpu_validation.py"]
expected_script = metadata["source"]["files"]["scripts/gpu_validation.py"]["sha256"]
if hashlib.sha256(bootstrap).hexdigest() != expected_script:
    raise ValueError("Bootstrap script does not match packaged source")
transport = Path("/content") / ("rlse-upload-" + uuid.uuid4().hex)
transport.mkdir(parents=True)
for name in required:
    (transport / name).write_bytes(uploaded[name])
root = Path("/content") / ("rlse-" + uuid.uuid4().hex)
subprocess.run([sys.executable, str(transport / "gpu_validation.py"), "unpack",
                "--archive", str(transport / "rlse-source.tar.gz"),
                "--sha256", metadata["archive_sha256"], "--destination", str(root)], check=True)
harness = root / "scripts/gpu_validation.py"
model_dir = Path("/content/rlse-models/Qwen3-0.6B")
cache_dir = Path("/content/rlse-target")
print("Verified source:", root)


## Pinned pretrained artifacts

This downloads `Qwen/Qwen3-0.6B` into `/content/rlse-models/Qwen3-0.6B` at the revision pinned in the independent generation fixture (`c1899de289a04d12100db370d81485cdf75e47ca`). All four artifact hashes must match; subsequent runs in the same Colab session reuse the verified artifacts without re-downloading 1.2 GB.


In [ ]:
download = subprocess.run([sys.executable, str(harness), "download-model", "--model-dir", str(model_dir)], check=False)
print("Artifact download exit code:", download.returncode)
# Continue to validation even after a download error so blocked prerequisites are recorded.


## CPU regressions, explicit CUDA inference, custom CUDA kernels, and live HTTP server validation

The harness records source/model identity, hardware, exact commands, exit codes, and logs. It runs:
1. Formatting (`cargo fmt --all --check`), CPU regressions (`cargo test --locked --all-targets`), and Python transport/evidence tests.
2. Pinned artifact hash verification.
3. CUDA Clippy and test builds (`--features cuda`).
4. Both explicit pretrained GPU gates (`test_gpu_inference` and `test_pretrained_quality`), requiring `ACTUAL_DEVICE=Cuda(...)`.
5. Custom CUDA kernel verification (`scripts/validate_cuda_kernels.py`: F32 & F16 multi-row Fused RMSNorm across dims `128, 1024`, plus F32 & F16 Ragged Batched Tiled Online-Softmax Paged GQA Decode Attention across boundary lengths `1, 15, 16, 17, 31, 32, 33, 64, 128`, fragmented page tables, and shared prefix page reuse).
6. Live HTTP GPU server validation (`scripts/validate_gpu_server.py`: concurrent warm/cold prefix continuous batching and mid-stream SSE Ctrl-C graceful draining).


In [ ]:
evidence = Path("/content") / ("rlse-evidence-" + uuid.uuid4().hex)
run = subprocess.run([
    sys.executable, str(harness), "run",
    "--device", "cuda",
    "--model-dir", str(model_dir),
    "--cache-dir", str(cache_dir),
    "--include-kernels",
    "--include-server",
    "--output", str(evidence),
], check=False)
results = json.loads((evidence / "results.json").read_text())
print(json.dumps(results, indent=2))
try:
    from google.colab import files
    files.download(str(evidence) + ".tar.gz")
    files.download(str(evidence) + ".tar.gz.json")
except Exception as exc:
    print(f"Evidence bundle saved on runtime at {evidence}.tar.gz (browser download skipped: {exc})")
if run.returncode:
    raise RuntimeError(f"Validation {results['status']}: {results.get('reason')}; inspect evidence")
